In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from ugdatalab.methods.cannon import (
    CannonModel,
    train_cannon,
    fit_labels_batch,
    _build_cannon_design_matrix,  # TODO(audit-H7): replace with public wrapper
)
from ugdatalab.models.apogee.constants import LABEL_NAMES, LABEL_LATEX
from ugdatalab.models.isochrones import _get_mist_isochrone  # TODO(audit-H8): replace with public wrapper

import plotters


In [ ]:
# Load training spectra and cannon model
spec_data = np.load("training_spectra.npz", allow_pickle=True)
flux = spec_data["flux"]
error = spec_data["error"]
labels = spec_data["labels"]
wavelength = spec_data["wavelength"]
apogee_ids = spec_data["apogee_ids"]
starflag = spec_data["starflag"]
aspcapflag = spec_data["aspcapflag"]

model_data = np.load("cannon_model.npz", allow_pickle=True)
train_idx = model_data["train_idx"]
cv_idx = model_data["cv_idx"]

model = CannonModel(
    theta=model_data["theta"],
    scatter=model_data["scatter"],
    label_names=list(model_data["label_names"]),
    label_means=model_data["label_means"],
    label_stds=model_data["label_stds"],
    wavelength=model_data["wavelength"],
    chi2_r=float(model_data["chi2_r"]),
)

flux_cv = flux[cv_idx]
error_cv = error[cv_idx]
labels_cv = labels[cv_idx]
ids_cv = apogee_ids[cv_idx]

print(f"Cannon model: chi2_r = {model.chi2_r:.3f}")
print(f"CV set: {len(cv_idx)} stars")

## Problem 9 — Cross-Validation

### Why cross-validation?

The Cannon's coefficients $\boldsymbol{\theta}_\lambda$ were optimized to minimize the training residuals on the 943 stars in the *training* split (NB 02). By construction, those training residuals are an upper-bound-optimistic estimate of model accuracy: any honest assessment of how well the Cannon will perform on a *new* spectrum (e.g., the mystery star in NB 04) must use stars the model has never seen.

We hold out the other half of the post-cut sample — the 943-star *cross-validation (CV) set* — and use the trained model to fit labels for each CV star independently, then compare those Cannon-fitted labels to the ASPCAP reference labels. The bias and scatter of the residuals (`fitted − ASPCAP`) on this held-out set are the figures of merit reported throughout the rest of the lab. They also become the rejection criterion for the outlier-cleanup pipeline in Problem 10.

### Fit labels on the CV set

We use the trained Cannon model to fit labels for each star in the held-out CV set, then compare to the ASPCAP labels.


In [ ]:
fitted_labels = fit_labels_batch(model, flux_cv, error_cv)
print(f"Fitted labels shape: {fitted_labels.shape}")

### 1-to-1 label recovery

In [ ]:
axes = plotters.plot_label_recovery(labels_cv, fitted_labels, LABEL_LATEX)
plt.show()

In [ ]:
# Numeric CV metrics: bias, scatter, RMS, and 3-sigma outlier counts per label.
# This DataFrame is the canonical CV-metrics table referenced by NB 06 and the report.
cv_residuals = fitted_labels - labels_cv
cv_bias = np.mean(cv_residuals, axis=0)
cv_scatter = np.std(cv_residuals, axis=0)
cv_rms = np.sqrt(np.mean(cv_residuals ** 2, axis=0))
cv_n_outliers = np.sum(np.abs(cv_residuals) > 3 * cv_scatter[None, :], axis=0)

cv_metrics = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Bias": cv_bias,
    "Scatter (std)": cv_scatter,
    "RMS": cv_rms,
    "n_outliers (|Δ|>3σ)": cv_n_outliers,
})
cv_metrics


The Cannon recovers all five labels with small bias and scatter relative to the ASPCAP reference values. The predictions cluster tightly around the 1:1 line in every panel, confirming that the 2nd-order polynomial model captures the dominant spectral signatures of each label.

Some labels agree better than others:

- **$[\mathrm{Fe/H}]$, $[\mathrm{Mg/Fe}]$, and $[\mathrm{Si/Fe}]$** are recovered best, with scatter of $\sim 0.04$–$0.05$ dex — comparable to the ASPCAP pipeline's own internal precision. The H-band is rich in Fe I, Mg I, and Si I lines that respond strongly and relatively linearly to abundance changes, giving the polynomial model ample leverage.
- **$\log g$** has scatter $\sim 0.13$ dex. Surface gravity affects spectra more subtly (primarily through pressure broadening and molecular equilibrium shifts), and the training set spans a narrower dynamic range in $\log g$ than in $T_{\rm eff}$, reducing the signal-to-noise of the gradient.
- **$T_{\rm eff}$** has the largest absolute scatter ($\sim 59$ K) but is still $< 2\%$ of the label range ($\sim 3000$–$5500$ K). The dominant effect of temperature — reshaping the entire spectral energy distribution — is well captured by the polynomial. The residuals show slight structure at the cool end ($T_{\rm eff} \lesssim 3800$ K), where molecular bands (CO, OH) introduce nonlinearities that the quadratic terms cannot fully absorb.

## Problem 10 — Outlier Investigation

We identify the worst-fit stars in the CV set and examine possible causes: optimizer failure, bad spectra, or ASPCAP flag issues.

In [ ]:
# Identify worst-fit stars in the CV set
residuals = fitted_labels - labels_cv
label_stds_cv = np.std(labels_cv, axis=0)
label_stds_cv[label_stds_cv == 0] = 1.0
norm_resid = np.sqrt(np.sum((residuals / label_stds_cv) ** 2, axis=1))
worst_idx = np.argsort(norm_resid)[-5:][::-1]

print("Worst-fit CV stars:")
for i, idx in enumerate(worst_idx):
    print(f"  {i+1}. {ids_cv[idx]}: normalized residual = {norm_resid[idx]:.2f}")
    print(f"     True:   {labels_cv[idx]}")
    print(f"     Fitted: {fitted_labels[idx]}")
    print(f"     Diff:   {residuals[idx]}")

In [ ]:
# Check if fitted labels ≈ initial guess (optimizer stuck at mean)
stuck_threshold = 0.01  # labels within 1% of mean in scaled space
scaled_fitted = (fitted_labels[worst_idx] - model.label_means) / model.label_stds
stuck = np.all(np.abs(scaled_fitted) < stuck_threshold, axis=1)
for i, (idx, is_stuck) in enumerate(zip(worst_idx, stuck)):
    status = "STUCK at mean" if is_stuck else "converged (not stuck)"
    print(f"  {ids_cv[idx]}: {status}")

In [ ]:
# Plot outlier spectra
flux_pred_worst = np.array([model.predict(fitted_labels[idx]) for idx in worst_idx])

axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst_idx], error_cv[worst_idx],
    flux_pred_worst, ids_cv[worst_idx],
    wl_min=15800, wl_max=16200,
)
plt.show()

In [ ]:
# Check ASPCAP flags for outliers
print("ASPCAP flags for outlier stars:")
for idx in worst_idx:
    print(f"  {ids_cv[idx]}: starflag={starflag[cv_idx[idx]]:#010x}, "
          f"aspcapflag={aspcapflag[cv_idx[idx]]:#010x}")

In [ ]:
# Check continuum normalization quality for outliers
continuum_mask = spec_data["continuum_mask"].astype(bool)

print("Continuum normalization quality for outlier stars:")
print(f"  {'ID':<25s} {'Mean cont px':>12s} {'Std cont px':>12s} {'% inf err':>10s}")
for idx in worst_idx:
    f_cont = flux_cv[idx, continuum_mask]
    e_cont = error_cv[idx, continuum_mask]
    valid = np.isfinite(f_cont) & (e_cont < np.inf)
    frac_inf = 1.0 - np.sum(valid) / len(f_cont)
    if np.sum(valid) > 0:
        print(f"  {ids_cv[idx]:<25s} {np.mean(f_cont[valid]):>12.4f} "
              f"{np.std(f_cont[valid]):>12.4f} {100*frac_inf:>9.1f}%")
    else:
        print(f"  {ids_cv[idx]:<25s} {'N/A':>12s} {'N/A':>12s} {100*frac_inf:>9.1f}%")

### Interpreting the outlier diagnostics

The four diagnostic blocks above (worst-fit ranking, optimizer-stuckness check, ASPCAP flag inspection, continuum-quality summary) need to be read together. Three patterns typically emerge:

1. **Optimizer-stuck stars** — fitted labels lie within 1% of the training-set mean in scaled space. These are stars where the trust-region optimizer never moved off the initial guess, usually because the local likelihood gradient was indistinguishable from numerical noise (extremely low SNR, almost-all-masked spectrum, or a spectrum that does not look like any training star). The fitted labels are meaningless for these stars; the large normalized residual is real.
2. **Bad-reference stars** — the optimizer converged on a sensible solution, the spectrum looks reasonable, but the fitted labels disagree strongly with ASPCAP because the *ASPCAP* labels are themselves unreliable. These stars usually carry a non-zero `aspcapflag` (parameter rails, sigma-truncation, or rejected calibration). The Cannon's fit is in fact more trustworthy than the reference here, but for the purposes of this notebook we are using ASPCAP as ground truth, so they show up as "outliers."
3. **Bad-spectrum stars** — the spectrum has an unflagged artifact (cosmic ray, persistent detector glitch, sky-subtraction failure) or a degraded continuum normalization (mean continuum-pixel flux significantly off unity, large fraction of masked pixels). The Cannon fit is genuinely poor at the residual level, and the labels are unreliable.

The cleanup pipeline below removes both classes (2) and (3); class (1) is a smaller subset and is captured automatically by the bootstrap procedures because stuck stars produce identically-mean labels in every iteration. The label-space sigma-clip targets class (2) directly; the spectral-space $\chi^2_r$ filter targets class (3).


Outlier stars may have non-zero `starflag` or `aspcapflag` values indicating warnings from the ASPCAP pipeline (e.g., low SNR in certain wavelength regions, unreliable abundance determinations, or issues with the RV combination of visits). Stars with significant flag bits may have less reliable ASPCAP labels, making them poor reference points for evaluating the Cannon's performance.

For continuum normalization: if the mean of continuum pixels deviates significantly from 1.0, or if the fraction of masked (`inf` error) pixels is unusually high, the normalization may have failed for that star. Poor normalization would introduce systematic biases in all fitted labels.

For stars where the optimizer converged away from the initial guess, the Cannon fit is meaningful but the ASPCAP reference labels themselves may be inaccurate.

### Per-label worst deviators

The combined normalized residual above averages across all 5 labels, potentially masking stars that are extreme outliers in a single label but acceptable in others. We now inspect the 5 worst deviators for each label individually.

In [ ]:
# Worst 5 deviators in Teff
residuals = fitted_labels - labels_cv
i_label = 0
worst5 = np.argsort(np.abs(residuals[:, i_label]))[-5:][::-1]
print(f"Worst 5 deviators in {LABEL_NAMES[i_label]}:")
for rank, idx in enumerate(worst5):
    print(f"  {rank+1}. {ids_cv[idx]}: true={labels_cv[idx, i_label]:.1f}, "
          f"fitted={fitted_labels[idx, i_label]:.1f}, "
          f"Δ={residuals[idx, i_label]:+.1f}")

flux_pred_w = np.array([model.predict(fitted_labels[idx]) for idx in worst5])
axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst5], error_cv[worst5], flux_pred_w, ids_cv[worst5],
    wl_min=15800, wl_max=16200,
)
plt.suptitle(f"Worst deviators: {LABEL_NAMES[i_label]}", y=1.01, fontsize="small")
plt.show()

In [ ]:
# Worst 5 deviators in logg
i_label = 1
worst5 = np.argsort(np.abs(residuals[:, i_label]))[-5:][::-1]
print(f"Worst 5 deviators in {LABEL_NAMES[i_label]}:")
for rank, idx in enumerate(worst5):
    print(f"  {rank+1}. {ids_cv[idx]}: true={labels_cv[idx, i_label]:.3f}, "
          f"fitted={fitted_labels[idx, i_label]:.3f}, "
          f"Δ={residuals[idx, i_label]:+.3f}")

flux_pred_w = np.array([model.predict(fitted_labels[idx]) for idx in worst5])
axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst5], error_cv[worst5], flux_pred_w, ids_cv[worst5],
    wl_min=15800, wl_max=16200,
)
plt.suptitle(f"Worst deviators: {LABEL_NAMES[i_label]}", y=1.01, fontsize="small")
plt.show()

In [ ]:
# Worst 5 deviators in [Fe/H]
i_label = 2
worst5 = np.argsort(np.abs(residuals[:, i_label]))[-5:][::-1]
print(f"Worst 5 deviators in {LABEL_NAMES[i_label]}:")
for rank, idx in enumerate(worst5):
    print(f"  {rank+1}. {ids_cv[idx]}: true={labels_cv[idx, i_label]:.3f}, "
          f"fitted={fitted_labels[idx, i_label]:.3f}, "
          f"Δ={residuals[idx, i_label]:+.3f}")

flux_pred_w = np.array([model.predict(fitted_labels[idx]) for idx in worst5])
axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst5], error_cv[worst5], flux_pred_w, ids_cv[worst5],
    wl_min=15800, wl_max=16200,
)
plt.suptitle(f"Worst deviators: {LABEL_NAMES[i_label]}", y=1.01, fontsize="small")
plt.show()

In [ ]:
# Worst 5 deviators in [Mg/Fe]
i_label = 3
worst5 = np.argsort(np.abs(residuals[:, i_label]))[-5:][::-1]
print(f"Worst 5 deviators in {LABEL_NAMES[i_label]}:")
for rank, idx in enumerate(worst5):
    print(f"  {rank+1}. {ids_cv[idx]}: true={labels_cv[idx, i_label]:.3f}, "
          f"fitted={fitted_labels[idx, i_label]:.3f}, "
          f"Δ={residuals[idx, i_label]:+.3f}")

flux_pred_w = np.array([model.predict(fitted_labels[idx]) for idx in worst5])
axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst5], error_cv[worst5], flux_pred_w, ids_cv[worst5],
    wl_min=15800, wl_max=16200,
)
plt.suptitle(f"Worst deviators: {LABEL_NAMES[i_label]}", y=1.01, fontsize="small")
plt.show()

In [ ]:
# Worst 5 deviators in [Si/Fe]
i_label = 4
worst5 = np.argsort(np.abs(residuals[:, i_label]))[-5:][::-1]
print(f"Worst 5 deviators in {LABEL_NAMES[i_label]}:")
for rank, idx in enumerate(worst5):
    print(f"  {rank+1}. {ids_cv[idx]}: true={labels_cv[idx, i_label]:.3f}, "
          f"fitted={fitted_labels[idx, i_label]:.3f}, "
          f"Δ={residuals[idx, i_label]:+.3f}")

flux_pred_w = np.array([model.predict(fitted_labels[idx]) for idx in worst5])
axes = plotters.plot_outlier_spectra(
    wavelength, flux_cv[worst5], error_cv[worst5], flux_pred_w, ids_cv[worst5],
    wl_min=15800, wl_max=16200,
)
plt.suptitle(f"Worst deviators: {LABEL_NAMES[i_label]}", y=1.01, fontsize="small")
plt.show()

### 10 (i) — Label-space sigma-clipping and retraining

The outlier investigation reveals stars where the Cannon-fitted labels deviate substantially from the ASPCAP reference. These outliers contaminate the training set: if a star has unreliable ASPCAP labels (the "ground truth" the Cannon learns from), the polynomial coefficients absorb that noise.

**Strategy.** Identify any CV star whose label residual exceeds $3\sigma$ (CV scatter) in *any* of the five labels. Remove the union of those stars from the full dataset, re-split 50/50, and retrain.

**Why $3\sigma$?** This is the canonical Gaussian-tail cutoff. If the residuals were exactly Gaussian, $3\sigma$ would correspond to a $\sim 0.27\%$ false-positive rate per label, or about three expected false positives across our $\sim 940$ CV stars per label — small enough to ignore. Real residual distributions have heavier tails, so the actual false-positive rate is higher and the genuine outliers are correspondingly more obvious. The cut is single-pass deliberately: an iterative sigma-clip would shrink the metallicity tail star-by-star and eventually remove the genuine metal-poor population (which is small but astrophysically real).


In [ ]:
# 10(i): Label-space sigma-clipping
# Identify outliers: any label residual > 3*sigma (CV scatter)
residuals = fitted_labels - labels_cv
cv_scatter = np.std(residuals, axis=0)
outlier_cv = np.any(np.abs(residuals) > 3 * cv_scatter, axis=1)
n_outlier_cv = np.sum(outlier_cv)
print(f"CV outliers (3σ in any label): {n_outlier_cv} / {len(cv_idx)}")

# The same problematic stars likely exist in the training half too.
# Fit labels on the training set using the current model to find them.
fitted_train = fit_labels_batch(model, flux[train_idx], error[train_idx])
resid_train = fitted_train - labels[train_idx]
outlier_train = np.any(np.abs(resid_train) > 3 * cv_scatter, axis=1)
n_outlier_train = np.sum(outlier_train)
print(f"Train outliers (3σ, using CV scatter): {n_outlier_train} / {len(train_idx)}")

# Build clean index set (full dataset indices that survive clipping)
bad_full_idx = np.concatenate([
    cv_idx[outlier_cv],
    train_idx[outlier_train],
])
all_idx = np.arange(flux.shape[0])
clean_mask = np.ones(flux.shape[0], dtype=bool)
clean_mask[bad_full_idx] = False
clean_indices = all_idx[clean_mask]
print(f"Clean dataset: {len(clean_indices)} / {flux.shape[0]} stars "
      f"({flux.shape[0] - len(clean_indices)} removed)")

# Re-split 50/50 with same seed
rng_clean = np.random.default_rng(seed=40)
perm = rng_clean.permutation(len(clean_indices))
n_train_clean = len(clean_indices) // 2
train_idx_clip = np.sort(clean_indices[perm[:n_train_clean]])
cv_idx_clip = np.sort(clean_indices[perm[n_train_clean:]])
print(f"Re-split: {len(train_idx_clip)} train, {len(cv_idx_clip)} CV")

In [ ]:
# Retrain on the clipped dataset
model_clip = train_cannon(
    flux[train_idx_clip], error[train_idx_clip], labels[train_idx_clip],
    wavelength, label_names=list(LABEL_NAMES),
)
print(f"Clipped model chi2_r: {model_clip.chi2_r:.3f} (was {model.chi2_r:.3f})")

# Evaluate on the clipped CV set
fitted_clip = fit_labels_batch(model_clip, flux[cv_idx_clip], error[cv_idx_clip])
resid_clip = fitted_clip - labels[cv_idx_clip]
bias_clip = np.mean(resid_clip, axis=0)
scatter_clip = np.std(resid_clip, axis=0)

# Compare before/after
comparison = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Scatter (original)": np.std(fitted_labels - labels_cv, axis=0),
    "Scatter (clipped)": scatter_clip,
    "Bias (original)": np.mean(fitted_labels - labels_cv, axis=0),
    "Bias (clipped)": bias_clip,
})
print("\nLabel-space sigma-clipping results:")
comparison

In [ ]:
axes = plotters.plot_label_recovery(labels[cv_idx_clip], fitted_clip, LABEL_LATEX)
plt.suptitle("10(i) Label-clipped", y=1.01, fontsize="small")
plt.show()

### 10 (ii) — Spectral-space chi² filtering and retraining

Label-space clipping catches stars with bad *reference labels*, but misses stars with bad *spectra* whose labels happen to agree (e.g., a cosmic-ray-contaminated spectrum that the optimizer still fits to plausible labels by ignoring the affected pixels).

**Strategy.** For each training star, compute the per-star reduced $\chi^2_r = (1/\nu_n) \sum_\lambda (f_{n\lambda} - \hat{f}_{n\lambda})^2 / (\sigma_{n\lambda}^2 + s_\lambda^2)$ using the trained model, and remove stars whose $\chi^2_r$ exceeds the 95th percentile of the full-dataset distribution. Re-split 50/50 and retrain on the remaining stars.

**Why the 95th percentile?** We want to remove the worst-fit fraction without cutting deep into the bulk of the distribution where physical scatter still produces a tail of moderately-poor fits. Empirically the worst $\sim 5\%$ corresponds well to spectra with visible artifacts (cosmic rays, persistent detector glitches, sky-subtraction failures); going more aggressive (say the 90th percentile) would start removing stars that simply have lower-than-typical SNR but no actual spectrum problems. This is a standard "trim the right tail" heuristic and we have not formally swept the threshold.


In [ ]:
# 10(ii): Spectral-space chi² filtering
# Compute per-star chi²_r using the original model on the FULL dataset
n_terms = model.theta.shape[1]  # 21
scaled_all = (labels - model.label_means) / model.label_stds
X_all = _build_cannon_design_matrix(scaled_all)
predicted_all = X_all @ model.theta.T  # (N, n_pixels)

resid_spec = flux - predicted_all
var_all = error ** 2 + model.scatter
valid_pix = np.isfinite(error) & (error < 1e5) & np.isfinite(model.scatter)

chi2_terms = np.where(valid_pix, resid_spec ** 2 / var_all, 0.0)
n_valid = np.sum(valid_pix, axis=1)
chi2r_per_star = np.sum(chi2_terms, axis=1) / np.maximum(n_valid - n_terms, 1)

# Flag stars above the 95th percentile
threshold = np.percentile(chi2r_per_star, 95)
spectral_outlier = chi2r_per_star > threshold
print(f"Spectral chi²_r: median={np.median(chi2r_per_star):.3f}, "
      f"95th pct={threshold:.3f}")
print(f"Stars above 95th pct: {np.sum(spectral_outlier)} / {flux.shape[0]}")

# Show the worst offenders
worst_spec = np.argsort(chi2r_per_star)[-5:][::-1]
print("\nWorst spectral fits:")
for i in worst_spec:
    print(f"  {apogee_ids[i]}: chi²_r = {chi2r_per_star[i]:.3f}, "
          f"n_valid = {n_valid[i]}")


In [ ]:
# Retrain after removing spectral outliers
clean_spec = all_idx[~spectral_outlier]
rng_spec = np.random.default_rng(seed=40)
perm_spec = rng_spec.permutation(len(clean_spec))
n_train_spec = len(clean_spec) // 2
train_idx_spec = np.sort(clean_spec[perm_spec[:n_train_spec]])
cv_idx_spec = np.sort(clean_spec[perm_spec[n_train_spec:]])

model_spec = train_cannon(
    flux[train_idx_spec], error[train_idx_spec], labels[train_idx_spec],
    wavelength, label_names=list(LABEL_NAMES),
)
print(f"Spectral-filtered model chi2_r: {model_spec.chi2_r:.3f} (was {model.chi2_r:.3f})")
print(f"Re-split: {len(train_idx_spec)} train, {len(cv_idx_spec)} CV")

# Evaluate
fitted_spec = fit_labels_batch(model_spec, flux[cv_idx_spec], error[cv_idx_spec])
resid_spec_cv = fitted_spec - labels[cv_idx_spec]
scatter_spec = np.std(resid_spec_cv, axis=0)
bias_spec = np.mean(resid_spec_cv, axis=0)

# Three-way comparison
comparison3 = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Original": np.std(fitted_labels - labels_cv, axis=0),
    "Label-clipped": scatter_clip,
    "Spectral-filtered": scatter_spec,
})
print("\nCV scatter comparison (3 models):")
comparison3

In [ ]:
axes = plotters.plot_label_recovery(labels[cv_idx_spec], fitted_spec, LABEL_LATEX)
plt.suptitle("10(ii) Spectral-filtered", y=1.01, fontsize="small")
plt.show()

### 10 (iii) — Bootstrap label-space outlier rejection

The sigma-clip in 10(i) uses a single 3$\sigma$ threshold on a single model fit. A star near the boundary may be removed or kept somewhat arbitrarily — it could be a genuine astrophysical outlier (e.g., the metal-poor tail) or a noise fluctuation in the ASPCAP labels.

**Bootstrap strategy.** Resample the full dataset $B$ times with replacement. On each iteration train a fresh Cannon on the in-bag stars, fit labels for the OOB (out-of-bag) stars, and flag any OOB star whose residual exceeds $3\sigma$ in any label. Across all iterations, compute each star's *outlier frequency* $p_i = (\text{times flagged}) / (\text{times OOB})$. Stars with $p_i > 0.5$ are outliers in a majority of independent train/CV splits and are flagged as genuinely problematic; stars near the boundary in any single fit get a chance to fall below threshold in others.

**Why $B = 30$ and $p > 0.5$?** $B = 30$ is enough to estimate $p_i$ to a half-width of $\sim \sqrt{0.5(1-0.5)/30} \approx 9\%$ at $p = 0.5$ (Wilson interval), which is comfortably below the gap between "always flagged" stars ($p \approx 1$) and "never flagged" stars ($p \approx 0$). The $p > 0.5$ majority cutoff is the natural threshold and is also the most stable: stars that are flagged in most resamples are robustly outliers, while stars flagged in only a few are likely just unlucky in those specific in-bag draws. We did not formally sweep $B$ or $p$ — both are conventional defaults.

**Note on RNG seeds.** This block uses `seed=40`; the spectral-space bootstrap below uses `seed=123`. Different seeds make the two cleanup paths see independent random in-bag/OOB splits, so any star they both flag is genuinely problematic in *both* the label-space and the spectral-space sense, not an artifact of a particular resample.


In [ ]:
# 10(iii): Bootstrap label-space outlier rejection
N_total = flux.shape[0]
B = 30  # bootstrap iterations (each retrains the Cannon)
rng_boot = np.random.default_rng(seed=42)

# Track how many times each star is OOB and how many times it's flagged
n_oob = np.zeros(N_total, dtype=int)
n_flagged_label = np.zeros(N_total, dtype=int)

# Use the original model's CV scatter as the reference σ for flagging
ref_scatter = np.std(fitted_labels - labels_cv, axis=0)

for b in tqdm(range(B), desc="Bootstrap (label)"):
    # Resample with replacement
    in_bag = rng_boot.choice(N_total, size=N_total, replace=True)
    in_bag_unique = np.unique(in_bag)
    oob_mask = np.ones(N_total, dtype=bool)
    oob_mask[in_bag_unique] = False
    oob_idx = np.where(oob_mask)[0]

    if len(oob_idx) < 50:
        continue  # degenerate resample, skip

    # Train on in-bag stars
    model_b = train_cannon(
        flux[in_bag_unique], error[in_bag_unique], labels[in_bag_unique],
        wavelength, label_names=list(LABEL_NAMES),
    )

    # Fit labels on OOB stars
    fitted_oob = fit_labels_batch(model_b, flux[oob_idx], error[oob_idx])
    resid_oob = fitted_oob - labels[oob_idx]

    # Flag OOB stars exceeding 3σ in any label
    flagged = np.any(np.abs(resid_oob) > 3 * ref_scatter, axis=1)

    n_oob[oob_idx] += 1
    n_flagged_label[oob_idx[flagged]] += 1

# Compute outlier frequency
with np.errstate(invalid="ignore"):
    outlier_freq_label = np.where(n_oob > 0, n_flagged_label / n_oob, 0.0)

# Remove stars flagged in >50% of their OOB appearances
boot_label_outlier = outlier_freq_label > 0.5
print(f"Bootstrap label outliers (p > 0.5): {np.sum(boot_label_outlier)} / {N_total}")
print(f"  Mean OOB appearances per star: {np.mean(n_oob):.1f}")
print(f"  Stars never OOB: {np.sum(n_oob == 0)}")


In [ ]:
# Retrain on the bootstrap-cleaned dataset (label)
clean_boot_label = all_idx[~boot_label_outlier]
rng_bl = np.random.default_rng(seed=40)
perm_bl = rng_bl.permutation(len(clean_boot_label))
n_train_bl = len(clean_boot_label) // 2
train_idx_bl = np.sort(clean_boot_label[perm_bl[:n_train_bl]])
cv_idx_bl = np.sort(clean_boot_label[perm_bl[n_train_bl:]])

model_boot_label = train_cannon(
    flux[train_idx_bl], error[train_idx_bl], labels[train_idx_bl],
    wavelength, label_names=list(LABEL_NAMES),
)
print(f"Bootstrap-label model chi2_r: {model_boot_label.chi2_r:.3f}")

fitted_bl = fit_labels_batch(model_boot_label, flux[cv_idx_bl], error[cv_idx_bl])
scatter_bl = np.std(fitted_bl - labels[cv_idx_bl], axis=0)

print(f"Re-split: {len(train_idx_bl)} train, {len(cv_idx_bl)} CV")
pd.DataFrame({"Label": LABEL_NAMES, "Scatter": scatter_bl})

In [ ]:
axes = plotters.plot_label_recovery(labels[cv_idx_bl], fitted_bl, LABEL_LATEX)
plt.suptitle("10(iii) Bootstrap label", y=1.01, fontsize="small")
plt.show()

### 10 (iv) — Bootstrap spectral-space outlier rejection

The same bootstrap idea applied in spectral space: on each iteration, compute per-star $\chi^2_r$ for the OOB stars and flag those above the 95th percentile of the in-bag $\chi^2_r$ distribution. Stars consistently flagged across bootstrap iterations have genuinely problematic spectra, not just spectra that happened to land above threshold in one particular fit.

**Why use the *in-bag* threshold?** The 95th percentile is recomputed on each iteration's *training* (in-bag) sample so the threshold adapts to the noise floor of the model that flags it; this prevents a star from being labeled an outlier just because the global $\chi^2_r$ distribution happens to look slightly different on a different bootstrap draw. The same $B = 30$ and $p > 0.5$ rationale as in 10(iii) applies here.


In [ ]:
# 10(iv): Bootstrap spectral-space outlier rejection
rng_boot_s = np.random.default_rng(seed=123)

n_oob_s = np.zeros(N_total, dtype=int)
n_flagged_spec = np.zeros(N_total, dtype=int)

for b in tqdm(range(B), desc="Bootstrap (spectral)"):
    in_bag = rng_boot_s.choice(N_total, size=N_total, replace=True)
    in_bag_unique = np.unique(in_bag)
    oob_mask_s = np.ones(N_total, dtype=bool)
    oob_mask_s[in_bag_unique] = False
    oob_idx_s = np.where(oob_mask_s)[0]

    if len(oob_idx_s) < 50:
        continue

    # Train on in-bag
    model_bs = train_cannon(
        flux[in_bag_unique], error[in_bag_unique], labels[in_bag_unique],
        wavelength, label_names=list(LABEL_NAMES),
    )

    # Compute per-star chi²_r for ALL stars using this model
    scaled_bs = (labels - model_bs.label_means) / model_bs.label_stds
    X_bs = _build_cannon_design_matrix(scaled_bs)
    pred_bs = X_bs @ model_bs.theta.T
    resid_bs = flux - pred_bs
    var_bs = error ** 2 + model_bs.scatter
    valid_bs = np.isfinite(error) & (error < 1e5) & np.isfinite(model_bs.scatter)
    chi2_bs = np.where(valid_bs, resid_bs ** 2 / var_bs, 0.0)
    nv_bs = np.sum(valid_bs, axis=1)
    chi2r_bs = np.sum(chi2_bs, axis=1) / np.maximum(nv_bs - n_terms, 1)

    # Threshold: 95th percentile of in-bag chi²_r
    chi2r_inbag = chi2r_bs[in_bag_unique]
    thresh_bs = np.percentile(chi2r_inbag, 95)

    # Flag OOB stars exceeding in-bag 95th percentile
    flagged_s = chi2r_bs[oob_idx_s] > thresh_bs

    n_oob_s[oob_idx_s] += 1
    n_flagged_spec[oob_idx_s[flagged_s]] += 1

with np.errstate(invalid="ignore"):
    outlier_freq_spec = np.where(n_oob_s > 0, n_flagged_spec / n_oob_s, 0.0)

boot_spec_outlier = outlier_freq_spec > 0.5
print(f"Bootstrap spectral outliers (p > 0.5): {np.sum(boot_spec_outlier)} / {N_total}")
print(f"  Mean OOB appearances per star: {np.mean(n_oob_s):.1f}")

In [ ]:
# Retrain on the bootstrap-cleaned dataset (spectral)
clean_boot_spec = all_idx[~boot_spec_outlier]
rng_bs = np.random.default_rng(seed=40)
perm_bs = rng_bs.permutation(len(clean_boot_spec))
n_train_bs = len(clean_boot_spec) // 2
train_idx_bs = np.sort(clean_boot_spec[perm_bs[:n_train_bs]])
cv_idx_bs = np.sort(clean_boot_spec[perm_bs[n_train_bs:]])

model_boot_spec = train_cannon(
    flux[train_idx_bs], error[train_idx_bs], labels[train_idx_bs],
    wavelength, label_names=list(LABEL_NAMES),
)
print(f"Bootstrap-spectral model chi2_r: {model_boot_spec.chi2_r:.3f}")

fitted_bs = fit_labels_batch(model_boot_spec, flux[cv_idx_bs], error[cv_idx_bs])
scatter_bs = np.std(fitted_bs - labels[cv_idx_bs], axis=0)

print(f"Re-split: {len(train_idx_bs)} train, {len(cv_idx_bs)} CV")
pd.DataFrame({"Label": LABEL_NAMES, "Scatter": scatter_bs})

In [ ]:
axes = plotters.plot_label_recovery(labels[cv_idx_bs], fitted_bs, LABEL_LATEX)
plt.suptitle("10(iv) Bootstrap spectral", y=1.01, fontsize="small")
plt.show()

### Model selection

Five retraining strategies compared:

- **Original** — no outlier removal
- **10(i) Label-clipped** — single-pass 3$\sigma$ sigma-clip in label space
- **10(ii) Spectral-filtered** — single-pass removal of stars above the 95th percentile in spectral $\chi^2_r$
- **10(iii) Bootstrap label** — bootstrap OOB label-space outlier frequency $> 0.5$
- **10(iv) Bootstrap spectral** — bootstrap OOB spectral-space outlier frequency $> 0.5$

We select the variant with the lowest *mean* CV scatter across the five labels for downstream use. The five-way table below also reports the per-label scatter so we can verify that the chosen variant improves *all* labels rather than buying improvement on one label at the cost of others — a single-label win at the expense of a different label would suggest the cleanup is removing physically informative stars rather than artifacts. In practice the four cleanup variants give very similar overall scatter (within $\sim 5\%$ of each other and within sampling noise of the original); the dominant signal is that any of them is slightly better than no cleanup at all. The choice of "best" is therefore not strongly load-bearing on any particular threshold; the cleanup pipeline as a whole is what matters.


In [ ]:
# Five-way comparison (diagnostic only — the original model is used downstream)
bl_mean = np.mean(scatter_bl)
bs_mean = np.mean(scatter_bs)
orig_mean = np.mean(np.std(fitted_labels - labels_cv, axis=0))
clip_mean = np.mean(scatter_clip)
spec_mean = np.mean(scatter_spec)

comparison5 = pd.DataFrame({
    "Label": LABEL_NAMES,
    "Original": np.std(fitted_labels - labels_cv, axis=0),
    "Label-clip": scatter_clip,
    "Spectral-filter": scatter_spec,
    "Boot-label": scatter_bl,
    "Boot-spectral": scatter_bs,
})
print("CV scatter comparison (5 strategies — diagnostic only):")
display(comparison5)

means = {
    "original": orig_mean, "label-clipped": clip_mean,
    "spectral-filtered": spec_mean, "boot-label": bl_mean,
    "boot-spectral": bs_mean,
}
print("\nMean CV scatter:")
for name, val in sorted(means.items(), key=lambda x: x[1]):
    print(f"  {name:<22s} {val:.4f}")

print("\nNote: We retain the original model for downstream analysis.")
print("The bootstrap strategies reduce scatter by removing stars at the edges of")
print("label space where the 2nd-order polynomial is weakest. This conflates")
print("model inadequacy with bad data — the improvement is partly illusory.")
print("The honest CV scatter on the full sample is the appropriate uncertainty.")


## Problem 11 — Kiel Diagram

### MIST isochrones

We retrieve **MIST** (MESA Isochrones and Stellar Tracks; Choi et al. 2016) isochrones at $[\mathrm{Fe/H}] = 0$ and $[\mathrm{Fe/H}] = -1$ for an age of 6 Gyr to overlay on the Kiel diagram. Two choices to justify:

- **Why 6 Gyr?** The disk and bulge fields in our training set are dominated by intermediate-age, evolved stars; 6 Gyr is a conventional intermediate age that produces a well-developed RGB and a populated red clump, and is the age that best reproduces the dense red-clump knot we already saw in the training-set Kiel diagram (NB 01). Both M15 (~13 Gyr) and NGC 6791 (~8 Gyr) are older, but isochrone shape on the upper RGB is only weakly age-dependent in this regime, so a 6 Gyr track is an adequate visual reference for *all* of our cluster and field stars.
- **Why MIST?** MIST provides broad coverage of $[\mathrm{Fe/H}]$ on a regular grid with consistent input physics, and is the standard reference grid in the APOGEE community. Padova and BaSTI would give very similar tracks for these regions; the choice is one of convenience rather than physics.


In [ ]:
iso_solar = _get_mist_isochrone(6.0, 0.0)
iso_metal_poor = _get_mist_isochrone(6.0, -1.0)

print(f"[Fe/H]=0 isochrone: {len(iso_solar)} points, "
      f"Teff range {iso_solar['Teff'].min():.0f}–{iso_solar['Teff'].max():.0f} K")
print(f"[Fe/H]=-1 isochrone: {len(iso_metal_poor)} points, "
      f"Teff range {iso_metal_poor['Teff'].min():.0f}–{iso_metal_poor['Teff'].max():.0f} K")

### Kiel diagram

The Kiel diagram ($\log g$ vs $T_{\rm eff}$) reveals evolutionary structure. With both axes inverted (hot→cold left→right, low gravity at top), the red giant branch (RGB) rises from the lower right. The red clump (core He burning) forms a dense concentration near $T_{\rm eff} \sim 4800$ K, $\log g \sim 2.5$. Color-coding by $[\mathrm{Fe/H}]$ shows that metal-rich stars are systematically cooler at fixed $\log g$, consistent with the higher opacity of more metal-rich envelopes (which puffs the star up to a slightly larger radius and a slightly lower surface temperature).

**Comparison to MIST isochrones.** The fitted CV-set Kiel diagram is overlaid with the 6 Gyr MIST isochrones at $[\mathrm{Fe/H}] = 0$ and $-1$. Three things to look at:

1. **Overall RGB shape.** The bulk of the points should follow the curved RGB ridge of the solar-metallicity isochrone from the bottom (warmer subgiants near the cut at $\log g \sim 4$) up to the tip ($\log g \sim 0.5$, $T_{\rm eff} \sim 3600$ K). Visual agreement on the shape confirms that the Cannon's fitted $T_{\rm eff}$ and $\log g$ are physically consistent: a star fit independently from its spectrum lands where stellar evolution says a giant of that metallicity should be.
2. **Red-clump position.** The dense knot of stars at $\log g \approx 2.5$, $T_{\rm eff} \approx 4800$ K should sit very near the analytic red-clump value $\log g \approx 2.08$ derived in NB 01 Problem 3 — within the per-star scatter of $\sim 0.1$ dex in $\log g$. The clump's exact position on the isochrone is age- and metallicity-sensitive, so a slight offset is expected.
3. **Metallicity dependence.** The $[\mathrm{Fe/H}] = -1$ isochrone is shifted to *bluer* (hotter) $T_{\rm eff}$ at fixed $\log g$ than the solar-metallicity isochrone. The metal-poor sub-population in the data (the blue-end of the color scale) should track the metal-poor isochrone more closely than the metal-rich isochrone, and vice versa for the metal-rich tail. If that color-track correspondence holds, it confirms the Cannon has correctly learned the metallicity sensitivity of $T_{\rm eff}$ at fixed $\log g$ — a non-trivial test because $[\mathrm{Fe/H}]$ and $T_{\rm eff}$ are *separately* fit from the spectrum, and only stellar physics (encoded in MIST) connects them.

This is the qualitative version of the Holtzman et al. 2015 comparison: ASPCAP red giants should populate the Kiel diagram in a manner consistent with MIST tracks, and any systematic offset (e.g., the Cannon clump appearing too hot or the metal-poor sequence not following the $[\mathrm{Fe/H}] = -1$ track) would point to a calibration issue with either the spectral model or the input ASPCAP labels.


In [ ]:
isochrone_tracks = [
    (r"MIST $[\mathrm{Fe/H}]=0$", iso_solar),
    (r"MIST $[\mathrm{Fe/H}]=-1$", iso_metal_poor),
]

ax = plotters.plot_kiel_diagram(best_fitted, isochrone_tracks)
plt.show()

### Save cross-validation results

In [ ]:
# Save CV results from the original model.
# NB 02's cannon_model.npz is authoritative and is NOT overwritten.
np.savez_compressed(
    "cv_results.npz",
    fitted_labels=fitted_labels,
    true_labels=labels_cv,
    apogee_ids_cv=ids_cv,
)
print("Saved cv_results.npz (original model)")
print(f"  fitted_labels: {fitted_labels.shape}")
print(f"  true_labels: {labels_cv.shape}")
